<a href="https://colab.research.google.com/github/SumitMaske8055/AI-Text-Summarizer/blob/main/Text_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments , T5ForConditionalGeneration
print("Imported")

Imported


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving samsum-test.csv to samsum-test.csv
Saving samsum-train.csv to samsum-train.csv
Saving samsum-validation.csv to samsum-validation.csv


In [ ]:
# Load each CSV file
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")
print("Data Loaded")


Data Loaded


In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n=4000 , random_state = 42).reset_index(drop=True)
val_data = val_data.sample(n=500 , random_state = 42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

In [ ]:
# Data Pre-processing

In [ ]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," " ,text)#lines
    text = re.sub(r"\s+"," " ,text)#spaces
    text = re.sub(r"<.*?>"," " ,text)# html tags <p> <h1>
    text = text.strip().lower()
    return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
#tokenize


In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length" , max_length = 512 ,truncation=True)
    targets = tokenizer(data["summary"], padding="max_length" , max_length = 150 ,truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as label
    return inputs

print("Executed")

Executed


In [ ]:
train_dataset = train_data.apply(tokenize , axis = 1).tolist()
val_dataset = val_data.apply(tokenize , axis = 1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
  # inputs ids - dialogue => token ids
  # i => Eos ,0 => padding
  # attention mask
  #labels - target => summary token

# Key Components
# Input IDs: Numerical representation of dialogue text after tokenization.

# Special Tokens:

# i → EOS (end of sequence)

# 0 → Padding

# Attention Mask: Binary mask (1 = attend, 0 = ignore padding).

# Labels: Target sequence (summary tokens), aligned with input IDs but shifted for teacher forcing.



In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:

type(train_dataset)
type(val_dataset)

list

In [ ]:
#Working with our model

In [ ]:
# NLP => generation task

from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained("t5-small")
tokenizer = T5Tokenizer.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
import torch

# Prefer CUDA GPU if available, otherwise fallback
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Training on GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Training on Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Training on CPU")

model.to(device)


Training on GPU: Tesla T4


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training  Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
#train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.641540,0.381548
2,0.396954,0.359762
3,0.373850,0.354639
4,0.361382,0.350411
5,0.355251,0.349225
6,0.351102,0.348715


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9133466237386068, metrics={'train_runtime': 1332.0223, 'train_samples_per_second': 18.018, 'train_steps_per_second': 2.252, 'total_flos': 3248203235328000.0, 'train_loss': 0.9133466237386068, 'epoch': 6.0})

In [ ]:
# model load => fine tune => save the model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_sumary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_sumary_model/tokenizer_config.json',
 './saved_sumary_model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_sumary_model")
tokenizer.from_pretrained("./saved_sumary_model")

OSError: Error no file named model.safetensors, or pytorch_model.bin, found in directory ./saved_sumary_model.

In [ ]:
## Test the core logic  for summarization

In [ ]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) #clean


  #tokenize
  inputs = tokenize(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  )

  #genarate the summary => token ids
  model.to(device)
  targets = model.generate(
      input_ids = inputs_["input_ids"],
      attention_mask = inputs_["attention_mask"],
      max_length = 150,
      num_beams = 4,
      length_penalty = 0.6,
      early_stopping = True

  )

  #token ids convert to summary => decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True) #EOS , SEP
  return summary


In [ ]:
print ("Cell Executed")